<a href="https://colab.research.google.com/github/yashaswiprasad/fake-news-detection/blob/main/Fake%20news%20detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
print("Checking uploaded files...")
print(os.listdir("/content"))


Checking uploaded files...
['.config', 'Fake.csv', 'True.csv', 'truthscan_vectorizer.pkl', 'streamlit.log', 'app.py', 'truthscan_model.pkl', 'sample_data']


In [ ]:
import pandas as pd

#Load fake and genuine news datasets
fake=pd.read_csv("/content/Fake.csv")
true=pd.read_csv("/content/True.csv")

#Assign labels
fake["label"]=0
true["label"]=1

#Combine datasets
data=pd.concat([fake,true],ignore_index=True)

#Display dataset information
print("TruthScan AI Dataset Loaded Succesfully!")

print("Fake news articles:",len(fake))
print("Genuine news articles:",len(true))
print("Total news articles:",len(data))

print("\nfirst five records:")
print(data.head())

TruthScan AI Dataset Loaded Succesfully!
Fake news articles: 23481
Genuine news articles: 21417
Total news articles: 44898

first five records:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  label  
0  December 31, 2017      0  
1  December 31, 2017      0  
2  December 30, 2017      0  
3  December 29

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Prepare news content
data["content"] = (
    data["title"].fillna("") + " " +
    data["text"].fillna("")
)

# Remove empty articles
data = data[data["content"].str.strip() != ""]

# Separate input and output
X = data["content"]
y = data["label"]

# Split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training data:", len(X_train))
print("Testing data:", len(X_test))

# Convert text into numerical features
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.7
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train the AI model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Make predictions
predictions = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, predictions)

print("\nTruthScan AI Training Completed!")
print("Model Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, predictions))

Training data: 35918
Testing data: 8980

TruthScan AI Training Completed!
Model Accuracy: 98.57 %

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4696
           1       0.98      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [ ]:
def predict_news(news):
    # Convert news into TF-IDF features
    news_vector = vectorizer.transform([news])

    # Predict the result
    prediction = model.predict(news_vector)[0]

    # Get prediction confidence
    probabilities = model.predict_proba(news_vector)[0]

    if prediction == 0:
        result = "FAKE NEWS"
    else:
        result = "GENUINE NEWS"

    confidence = max(probabilities) * 100

    print("TRUTHSCAN AI RESULT")
    print("-------------------")
    print("Prediction:", result)
    print("Model confidence:", round(confidence, 2), "%")

# Test the prediction system
sample_news = input("Enter a news article: ")

predict_news(sample_news)

Enter a news article: The government announced a new education policy aimed at improving digital learning facilities in schools across the country. The policy includes funding for computers, internet access and teacher training.
TRUTHSCAN AI RESULT
-------------------
Prediction: FAKE NEWS
Model confidence: 54.8 %


In [ ]:
!pip install streamlit



In [ ]:
import pickle

with open("truthscan_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("truthscan_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("TruthScan AI model saved successfully!")

TruthScan AI model saved successfully!


In [ ]:
%%writefile app.py

import streamlit as st
import pickle

# Load trained AI model
with open("truthscan_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("truthscan_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

# Website configuration
st.set_page_config(
    page_title="TruthScan AI",
    page_icon="🔍",
    layout="centered"
)

# Website heading
st.title("🔍 TruthScan AI")
st.subheader("Fake News Detection System")

st.write(
    "Enter a news article below to check "
    "whether our AI model classifies it as Fake or Genuine."
)

# News input
news = st.text_area(
    "Enter your news article:",
    height=200,
    placeholder="Paste your news article here..."
)

# Prediction button
if st.button("Check News", type="primary"):

    if news.strip():

        news_vector = vectorizer.transform([news])
        prediction = model.predict(news_vector)[0]
        probabilities = model.predict_proba(news_vector)[0]

        confidence = max(probabilities) * 100

        if prediction == 0:
            st.error("🚨 FAKE NEWS")
        else:
            st.success("✅ GENUINE NEWS")

        st.write(
            f"Model confidence: {confidence:.2f}%"
        )

        st.caption(
            "This is an AI model prediction, "
            "not independent verification of the news."
        )

    else:
        st.warning("Please enter a news article first.")

Overwriting app.py


In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &

from google.colab.output import eval_js

print("Your TruthScan AI website is starting...")

url = eval_js("google.colab.kernel.proxyPort(8501)")

print("Open your website here:")
print(url)

Your TruthScan AI website is starting...
Open your website here:
https://8501-m-s-use1c2-3ew4sehld7sv6-c.us-east1.prod.r.colab.dev


In [ ]:
!tail -n 40 /content/streamlit.log


2026-11-17 17:31:45.724 Port 8501 is not available
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [ ]:
!streamlit run app.py --server.port 8502 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit2.log 2>&1 &

import time
from google.colab import output

time.sleep(5)

print("TruthScan AI is starting...")

url = output.eval_js("google.colab.kernel.proxyPort(8502)")

print("Open your website:")
print(url)

TruthScan AI is starting...
Open your website:
https://8502-m-s-use1c2-3ew4sehld7sv6-c.us-east1.prod.r.colab.dev


In [ ]:
%%writefile app.py

import streamlit as st
import pickle
import pandas as pd

st.set_page_config(
    page_title="TruthScan AI Dashboard",
    page_icon="🔍",
    layout="wide"
)

# Load trained model
with open("truthscan_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("truthscan_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

# Session history
if "history" not in st.session_state:
    st.session_state.history = []

# Sidebar
st.sidebar.title("🔍 TruthScan AI")
page = st.sidebar.radio(
    "Navigation",
    ["Dashboard", "News Detection", "Analytics", "History", "About"]
)

st.title("🔍 TruthScan AI")
st.caption("AI-Powered Fake News Detection System")

if page == "Dashboard":

    st.header("Dashboard Overview")

    total = len(st.session_state.history)
    fake = sum(x["Prediction"] == "FAKE" for x in st.session_state.history)
    genuine = total - fake

    c1, c2, c3 = st.columns(3)

    c1.metric("Articles Analysed", total)
    c2.metric("Fake News", fake)
    c3.metric("Genuine News", genuine)

    st.info("Use News Detection to analyse a news article.")

elif page == "News Detection":

    st.header("Check Your News")

    news = st.text_area(
        "Paste your news article",
        height=200
    )

    if st.button("Check News", type="primary"):

        if news.strip():

            features = vectorizer.transform([news])
            prediction = model.predict(features)[0]
            probabilities = model.predict_proba(features)[0]

            confidence = max(probabilities) * 100

            result = "FAKE" if prediction == 0 else "GENUINE"

            if result == "FAKE":
                st.error("🚨 Model Prediction: FAKE NEWS")
            else:
                st.success("✅ Model Prediction: GENUINE NEWS")

            st.metric("Model Confidence", f"{confidence:.2f}%")

            st.caption(
                "Model prediction only; this is not independent fact verification."
            )

            st.session_state.history.append({
                "Article": news[:100],
                "Prediction": result,
                "Confidence": round(confidence, 2)
            })

        else:
            st.warning("Please enter a news article.")

elif page == "Analytics":

    st.header("News Detection Analytics")

    history = pd.DataFrame(st.session_state.history)

    if not history.empty:

        st.subheader("Fake vs Genuine Predictions")

        counts = history["Prediction"].value_counts()

        st.bar_chart(counts)

        st.subheader("Prediction History")
        st.dataframe(history, use_container_width=True)

    else:
        st.info("No articles analysed yet. Visit News Detection first.")

elif page == "History":

    st.header("Previously Analysed News")

    history = pd.DataFrame(st.session_state.history)

    if not history.empty:
        st.dataframe(history, use_container_width=True)
    else:
        st.info("No prediction history available.")

elif page == "About":

    st.header("About TruthScan AI")

    st.write("""
    TruthScan AI is a machine-learning project
    designed to classify news articles as fake
    or genuine using TF-IDF and Logistic Regression.

    The system analyses text and provides a
    predicted classification with model confidence.

    Note: Predictions are not independent fact checks.
    """)

Overwriting app.py


In [ ]:
%%writefile app.py

import streamlit as st
import pickle
import pandas as pd

st.set_page_config(
    page_title="TruthScan AI Dashboard",
    page_icon="🔍",
    layout="wide"
)

# Load trained model
with open("truthscan_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("truthscan_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

# Session history
if "history" not in st.session_state:
    st.session_state.history = []

# Sidebar
st.sidebar.title("🔍 TruthScan AI")
page = st.sidebar.radio(
    "Navigation",
    ["Dashboard", "News Detection", "Analytics", "History", "About"]
)

st.title("🔍 TruthScan AI")
st.caption("AI-Powered Fake News Detection System")

if page == "Dashboard":

    st.header("Dashboard Overview")

    total = len(st.session_state.history)
    fake = sum(x["Prediction"] == "FAKE" for x in st.session_state.history)
    genuine = total - fake

    c1, c2, c3 = st.columns(3)

    c1.metric("Articles Analysed", total)
    c2.metric("Fake News", fake)
    c3.metric("Genuine News", genuine)

    st.info("Use News Detection to analyse a news article.")

elif page == "News Detection":

    st.header("Check Your News")

    news = st.text_area(
        "Paste your news article",
        height=200
    )

    if st.button("Check News", type="primary"):

        if news.strip():

            features = vectorizer.transform([news])
            prediction = model.predict(features)[0]
            probabilities = model.predict_proba(features)[0]

            confidence = max(probabilities) * 100

            result = "FAKE" if prediction == 0 else "GENUINE"

            if result == "FAKE":
                st.error("🚨 Model Prediction: FAKE NEWS")
            else:
                st.success("✅ Model Prediction: GENUINE NEWS")

            st.metric("Model Confidence", f"{confidence:.2f}%")

            st.caption(
                "Model prediction only; this is not independent fact verification."
            )

            st.session_state.history.append({
                "Article": news[:100],
                "Prediction": result,
                "Confidence": round(confidence, 2)
            })

        else:
            st.warning("Please enter a news article.")

elif page == "Analytics":

    st.header("News Detection Analytics")

    history = pd.DataFrame(st.session_state.history)

    if not history.empty:

        st.subheader("Fake vs Genuine Predictions")

        counts = history["Prediction"].value_counts()

        st.bar_chart(counts)

        st.subheader("Prediction History")
        st.dataframe(history, use_container_width=True)

    else:
        st.info("No articles analysed yet. Visit News Detection first.")

elif page == "History":

    st.header("Previously Analysed News")

    history = pd.DataFrame(st.session_state.history)

    if not history.empty:
        st.dataframe(history, use_container_width=True)
    else:
        st.info("No prediction history available.")

elif page == "About":

    st.header("About TruthScan AI")

    st.write("""
    TruthScan AI is a machine-learning project
    designed to classify news articles as fake
    or genuine using TF-IDF and Logistic Regression.

    The system analyses text and provides a
    predicted classification with model confidence.

    Note: Predictions are not independent fact checks.
    """)

Overwriting app.py


In [ ]:
!streamlit run app.py --server.port 8503 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit3.log 2>&1 &

import time
from google.colab import output

time.sleep(5)

url = output.eval_js("google.colab.kernel.proxyPort(8503)")

print("TruthScan AI Dashboard:")
print(url)

TruthScan AI Dashboard:
https://8503-m-s-usc1b2-2pgr4gdl9dijz-b.us-central1.prod.r.colab.dev


In [ ]:
!streamlit run app.py --server.port 8503 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit3.log 2>&1 &

import time
from google.colab import output

time.sleep(5)

url = output.eval_js("google.colab.kernel.proxyPort(8503)")

print("TruthScan AI Dashboard:")
print(url)

TruthScan AI Dashboard:
https://8503-m-s-usc1b2-2pgr4gdl9dijz-b.us-central1.prod.r.colab.dev


In [ ]:
!tail -n 25 /content/streamlit3.log

/bin/bash: line 1: streamlit: command not found


In [ ]:
!python -m pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 26.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 59.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [ ]:
import subprocess
import sys
import time
from google.colab import output

# Start TruthScan AI on a fresh port
log = open("/content/streamlit4.log", "w")

process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", "8504",
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false"
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(8)

url = output.eval_js(
    "google.colab.kernel.proxyPort(8504)"
)

print("TruthScan AI Dashboard:")
print(url)

TruthScan AI Dashboard:
https://8504-m-s-usc1b2-2pgr4gdl9dijz-b.us-central1.prod.r.colab.dev


In [ ]:
!tail -n 40 /content/streamlit4.log

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [ ]:
import os

print("Current folder:", os.getcwd())

print("\nFiles in /content:")
print(os.listdir("/content"))

Current folder: /content

Files in /content:
['.config', 'streamlit3.log', 'streamlit4.log', 'sample_data']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder = "/content/drive/MyDrive"

for root, dirs, files in os.walk(folder):
    for name in files:
        if name in [
            "app.py",
            "truthscan_model.pkl",
            "truthscan_vectorizer.pkl",
            "Fake.csv",
            "True.csv"
        ]:
            print(os.path.join(root, name))

ValueError: mount failed

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder = "/content/drive/MyDrive"

for root, dirs, files in os.walk(folder):
    for name in files:
        if name in [
            "app.py",
            "truthscan_model.pkl",
            "truthscan_vectorizer.pkl",
            "Fake.csv",
            "True.csv"
        ]:
            print(os.path.join(root, name))

MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder = "/content/drive/MyDrive"

for root, dirs, files in os.walk(folder):
    for name in files:
        if name in [
            "app.py",
            "truthscan_model.pkl",
            "truthscan_vectorizer.pkl",
            "Fake.csv",
            "True.csv"
        ]:
            print(os.path.join(root, name))

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os

print("Current folder:", os.getcwd())
print("Available files:", os.listdir("/content"))

Current folder: /content
Available files: ['.config', 'sample_data']


In [ ]:
import os
print(os.listdir("/content"))

['.config', 'sample_data']


In [ ]:
import pandas as pd

fake = pd.read_csv("/content/Fake.csv")
true = pd.read_csv("/content/True.csv")

print("Fake news records:", len(fake))
print("Genuine news records:", len(true))

print("\nFake news sample:")
display(fake.head())

print("\nGenuine news sample:")
display(true.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/Fake.csv'

In [ ]:
import os

print("Files in root folder:")
print(os.listdir("/"))

print("\nFiles in content folder:")
print(os.listdir("/content"))

Files in root folder:
['boot', 'bin', 'dev', 'srv', 'usr', 'proc', 'run', 'sys', 'var', 'home', 'etc', 'media', 'mnt', 'lib', 'root', 'opt', 'sbin', 'tmp', 'lib64', 'True.csv', 'Fake.csv', 'kaggle', '.dockerenv', 'backend-container', 'datalab', 'tools', 'content', 'python-apt', 'python-apt.tar.xz', 'bin.usr-is-merged', 'sbin.usr-is-merged', 'lib.usr-is-merged']

Files in content folder:
['.config', 'sample_data']


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Fake.csv to Fake.csv


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving True.csv to True.csv


In [ ]:
import pandas as pd

fake = pd.read_csv("/content/Fake.csv")
true = pd.read_csv("/content/True.csv")

print("Fake news records:", len(fake))
print("Genuine news records:", len(true))

print("\nFake News Sample:")
display(fake.head())

print("\nGenuine News Sample:")
display(true.head())

Fake news records: 23481
Genuine news records: 21417

Fake News Sample:


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"



Genuine News Sample:


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

fake = pd.read_csv("/content/Fake.csv")
true = pd.read_csv("/content/True.csv")

fake["label"] = 0
true["label"] = 1

data = pd.concat([fake, true], ignore_index=True)

data["text"] = data["title"].fillna("") + " " + data["text"].fillna("")

X = data["text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

predictions = model.predict(X_test_vec)

print("Model Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Model Accuracy: 0.987305122494432
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4696
           1       0.98      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [ ]:
import pickle

# Save trained model
with open("/content/truthscan_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save text vectorizer
with open("/content/truthscan_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("TruthScan AI model saved successfully!")

TruthScan AI model saved successfully!


In [ ]:
%%writefile app.py
import streamlit as st
import pickle

st.set_page_config(
    page_title="TruthScan AI",
    page_icon="🔎",
    layout="wide"
)

with open("/content/truthscan_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("/content/truthscan_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

st.title("🔎 TruthScan AI")
st.subheader("AI-Powered Fake News Detection System")

st.write("Enter a news article below to analyse it.")

news = st.text_area(
    "Paste your news article here:",
    height=200
)

if st.button("Analyse News"):
    if news.strip():
        transformed = vectorizer.transform([news])
        prediction = model.predict(transformed)[0]
        probability = model.predict_proba(transformed)[0]

        if prediction == 0:
            st.error("⚠️ Prediction: FAKE NEWS")
        else:
            st.success("✅ Prediction: GENUINE NEWS")

        st.write(
            f"Model confidence: "
            f"{max(probability) * 100:.2f}%"
        )
        st.caption(
            "This is an AI model prediction, not independent "
            "verification of the article."
        )
    else:
        st.warning("Please enter a news article.")

st.divider()
st.caption("TruthScan AI | Fake News Detection Project")

Writing app.py


In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 12.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 48.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [ ]:
!streamlit run app.py &>/content/streamlit.log &

In [ ]:
from google.colab import output
import time

time.sleep(5)

url = output.eval_js(
    "google.colab.kernel.proxyPort(8501)"
)

print("TruthScan AI Dashboard:")
print(url)

TruthScan AI Dashboard:
https://8501-m-s-use1d13-2rwltqvadiy6m-d.us-east1.prod.r.colab.dev


In [16]:
!tail -n 40 /content/streamlit.log

2026-11-18 14:46:03.640 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-use1d13-2rwltqvadiy6m-d.us-east1.prod.r.colab.dev, host=m-s-use1d13-2rwltqvadiy6m.us-east1-d.c.codatalab-user-runtimes.internal:8007
2026-11-18 14:46:11.694 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-use1d13-2rwltqvadiy6m-d.us-east1.prod.r.colab.dev, host=m-s-use1d13-2rwltqvadiy6m.us-east1-d.c.codatalab-user-runtimes.internal:8007
2026-11-18 14:46:19.393 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-use1d13-2rwltqvadiy6m-d.us-east1.prod.r.colab.dev, host=m-s-use1d13-2rwltqvadiy6m.us-east1-d.c.codatalab-user-runtimes.internal:8007
2026-11-18 14:46:27.000 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-use1d13-2rwltqvadiy6m-d.us-east1.prod.r.colab.dev, host=m-s-use1d13-2rwltqvadiy6m.us-east1-d.c.codatalab-user-runtimes.internal:8

In [17]:
!pkill -f streamlit

In [18]:
!streamlit run /content/app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /content/streamlit.log 2>&1 &

In [19]:
from google.colab import output
import time

time.sleep(5)

url = output.eval_js(
    "google.colab.kernel.proxyPort(8501)"
)

print("TruthScan AI Dashboard:")
print(url)

TruthScan AI Dashboard:
https://8501-m-s-use1d13-2rwltqvadiy6m-d.us-east1.prod.r.colab.dev


In [20]:
text = """NASA launched the James Webb Space Telescope
on December 25, 2021. It was developed through an
international partnership involving NASA, ESA and CSA."""

vector = vectorizer.transform([text])

print("Prediction:", model.predict(vector)[0])
print("Probabilities:", model.predict_proba(vector)[0])

Prediction: 0
Probabilities: [0.72683457 0.27316543]


In [21]:
print("Dataset distribution:")
print(data["label"].value_counts())

print("\nMissing text values:")
print(data["text"].isna().sum())

print("\nModel classes:")
print(model.classes_)

Dataset distribution:
label
0    23481
1    21417
Name: count, dtype: int64

Missing text values:
0

Model classes:
[0 1]


In [22]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.987305122494432

Confusion Matrix:
[[4623   73]
 [  41 4243]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4696
           1       0.98      0.99      0.99      4284

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980

